<a href="https://colab.research.google.com/github/trinhtattran/RAGassistant/blob/main/PD_RAG_Assistant.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [10]:
DATA_DIR = "/content/drive/MyDrive/RAG/data"

In [11]:
!pip -q install langchain langchain-community sentence-transformers faiss-cpu pypdf

In [12]:
!pip install -q pypdf==3.17.4

In [13]:
!pip install -q pdfplumber


In [14]:
import langchain
import sentence_transformers
import faiss
import pypdf

print("All imports OK")


All imports OK


In [15]:
import os, re
from langchain_community.document_loaders import PDFPlumberLoader #more tables on my docs now

DATA_DIR = "/content/drive/MyDrive/RAG/data"

def load_pdfs(folder_path):
    docs = []
    for fn in os.listdir(folder_path):
        if fn.lower().endswith(".pdf"):
            loader = PDFPlumberLoader(os.path.join(folder_path, fn))
            loaded = loader.load()
            for d in loaded:
                d.metadata["source"] = fn
            docs.extend(loaded)
    return docs

def basic_clean(text: str) -> str:
    text = re.sub(r"\s+", " ", text)
    return text.strip()

raw_docs = load_pdfs(DATA_DIR)

for d in raw_docs:
    d.page_content = basic_clean(d.page_content)

print("Loaded pages:", len(raw_docs))
print("Example source:", raw_docs[0].metadata)
print("Example text:", raw_docs[0].page_content[:400])


Loaded pages: 615
Example source: {'source': 'Parkinson’s Disease Pathogenesis and Clinical Aspects.pdf', 'file_path': '/content/drive/MyDrive/RAG/data/Parkinson’s Disease Pathogenesis and Clinical Aspects.pdf', 'page': 0, 'total_pages': 194, 'CreationDate': "D:20181224134218+05'30'", 'Creator': 'Adobe InDesign CS5.5 (7.5)', 'ModDate': "D:20181224134252+05'30'", 'Producer': 'Adobe PDF Library 9.9', 'Trapped': 'False'}
Example text: Parkinson’s Disease Pathogenesis and Clinical Aspects Cover image: A case of Parkinson’s disease as described and illustrated by William Gowers. See page 112, Chapter 6 for details. CP-005.indb 1 24/12/18 1:42 PM


In [16]:
print("Loaded pages:", len(raw_docs))
print("Sample text length:", len(raw_docs[0].page_content))
print(raw_docs[0].page_content[:500])
#testing to see if docs are read

Loaded pages: 615
Sample text length: 212
Parkinson’s Disease Pathogenesis and Clinical Aspects Cover image: A case of Parkinson’s disease as described and illustrated by William Gowers. See page 112, Chapter 6 for details. CP-005.indb 1 24/12/18 1:42 PM


In [17]:
from collections import defaultdict
import numpy as np

# raw_docs is a list of LangChain Documents (one per page)
lengths = [len(d.page_content.strip()) for d in raw_docs]

print("Total pages:", len(raw_docs))
print("Pages with 0 chars:", sum(l == 0 for l in lengths))
print("Pages with <50 chars:", sum(l < 50 for l in lengths))
print("Median chars/page:", int(np.median(lengths)))
print("10th percentile chars/page:", int(np.percentile(lengths, 10)))

# Group by PDF filename
by_pdf = defaultdict(list)
for d in raw_docs:
    by_pdf[d.metadata.get("source","UNKNOWN")].append(len(d.page_content.strip()))

print("\nWorst PDFs by median extracted chars/page:")
stats = []
for pdf, lens in by_pdf.items():
    stats.append((pdf, int(np.median(lens)), sum(l == 0 for l in lens), len(lens)))
stats.sort(key=lambda x: x[1])  # sort by median text length

for pdf, med, zeros, total in stats[:10]:
    print(f"{pdf:45}  median={med:4d}  zero_pages={zeros:3d}/{total}")


Total pages: 615
Pages with 0 chars: 23
Pages with <50 chars: 31
Median chars/page: 3075
10th percentile chars/page: 675

Worst PDFs by median extracted chars/page:
Parkinsonism vs Parkinson’s disease.pdf        median= 999  zero_pages=  0/8
Atypical Parkinsonism BCM.pdf                  median=1308  zero_pages=  0/8
Atypical Parkinsonism.pdf                      median=1435  zero_pages=  0/11
Clinical effectiveness and cost-effectiveness of physiotherapy and occupational therapy versus no therapy in mild to moderate Parkinson’s disease.pdf  median=1631  zero_pages= 23/124
UPDRS.pdf                                      median=1745  zero_pages=  0/8
Atypical Parkinsonian Disorders.pdf            median=2202  zero_pages=  0/4
MDS-UPDRS.pdf                                  median=2408  zero_pages=  0/33
How to approach a patient with parkinsonism - red flags for atypical parkinsonism.pdf  median=2432  zero_pages=  0/34
Stages in Parkinson’s Disease.pdf              median=2829  zero_pages

In [18]:
from langchain_text_splitters import RecursiveCharacterTextSplitter


splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=120
)

chunks = splitter.split_documents(raw_docs)

print("Total chunks:", len(chunks))
print("One chunk metadata:", chunks[0].metadata)
print("One chunk text:", chunks[0].page_content[:300]) #need to print out to see


Total chunks: 3054
One chunk metadata: {'source': 'Parkinson’s Disease Pathogenesis and Clinical Aspects.pdf', 'file_path': '/content/drive/MyDrive/RAG/data/Parkinson’s Disease Pathogenesis and Clinical Aspects.pdf', 'page': 0, 'total_pages': 194, 'CreationDate': "D:20181224134218+05'30'", 'Creator': 'Adobe InDesign CS5.5 (7.5)', 'ModDate': "D:20181224134252+05'30'", 'Producer': 'Adobe PDF Library 9.9', 'Trapped': 'False'}
One chunk text: Parkinson’s Disease Pathogenesis and Clinical Aspects Cover image: A case of Parkinson’s disease as described and illustrated by William Gowers. See page 112, Chapter 6 for details. CP-005.indb 1 24/12/18 1:42 PM


In [19]:
import random

def print_random_chunks(chunks, n=5, max_chars=900):
    for idx in random.sample(range(len(chunks)), n):
        c = chunks[idx]
        print("\n" + "="*90)
        print(f"CHUNK #{idx}")
        print("SOURCE:", c.metadata.get("source"), "| PAGE:", c.metadata.get("page"))
        print(c.page_content[:max_chars])

print_random_chunks(chunks, n=5)
#this size works!


CHUNK #380
SOURCE: Parkinson’s Disease Pathogenesis and Clinical Aspects.pdf | PAGE: 85
example, it can promote mitochondrial biogenesis, mtDNA replication, and transcription of mitochondrial genes (107). Thus, Parkin is vital for mitochon- drial respiration and function (107). In addition, Parkin acts as an E3 ubiquitin protein ligase that targets particular substrates for degradation via the ubiquitin- proteasome system, including the glycoslyated form of α-synuclein (108). The loss of Parkin activity is thought to contribute to the buildup of toxic protein aggregates causing Parkinson’s disease (108). Interestingly, Parkin acts down- stream of one of the other aforementioned genes—PINK1—a mitochondrial kinase in which mutations can cause an autosomal recessive familial form of early onset Parkinson’s disease. This is demonstrated by the fact that Parkin over- expression

CHUNK #93
SOURCE: Parkinson’s Disease Pathogenesis and Clinical Aspects.pdf | PAGE: 29
individuals (146–149). Fu

In [24]:
!pip -q install chromadb
import chromadb
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction

CHROMA_PATH = "/content/drive/MyDrive/RAG/chroma_db"  # persistent storage
client = chromadb.PersistentClient(path=CHROMA_PATH)

embed_fn = SentenceTransformerEmbeddingFunction(model_name="sentence-transformers/all-MiniLM-L6-v2")

collection = client.get_or_create_collection(
    name="pd_rag_chunks",
    embedding_function=embed_fn
)

print("Chroma collection ready. Current count:", collection.count())


Chroma collection ready. Current count: 0


In [25]:
# Convert LangChain Documents -> lists Chroma expects
ids = [f"chunk_{i:05d}" for i in range(len(chunks))]
documents = [c.page_content for c in chunks]

metadatas = []
for c in chunks:
    metadatas.append({
        "source": c.metadata.get("source"),
        "page": c.metadata.get("page"),
        "modality": "text"  # optional, recommended in slides for filtering
    })

# If you rerun this cell, Chroma will complain about duplicate IDs.
# To be safe, delete and re-add if the collection isn't empty:
if collection.count() > 0:
    collection.delete(ids=collection.get(include=[]).get("ids", []))  # safe reset for small projects

collection.add(ids=ids, documents=documents, metadatas=metadatas)
print("Inserted into Chroma. New count:", collection.count())


Inserted into Chroma. New count: 3054


In [26]:
def retrieve_top_k(query: str, k: int = 6):
    res = collection.query(query_texts=[query], n_results=k)
    # res fields are lists-of-lists because it supports multiple queries at once
    hits = []
    for doc, meta, dist in zip(res["documents"][0], res["metadatas"][0], res["distances"][0]):
        hits.append({"text": doc, "metadata": meta, "distance": dist})
    return hits

def show_hits(hits, max_chars=450):
    for i, h in enumerate(hits, 1):
        m = h["metadata"]
        print("\n" + "-"*90)
        print(f"HIT {i} | source={m.get('source')} | page={m.get('page')} | distance={h['distance']:.4f}")
        print(h["text"][:max_chars])


In [27]:
hits = retrieve_top_k("What are red flags for atypical parkinsonism?", k=6)
show_hits(hits)



------------------------------------------------------------------------------------------
HIT 1 | source=Parkinson’s Disease Pathogenesis and Clinical Aspects.pdf | page=142 | distance=0.3297
Aspects. Stoker TB, Greenland JC (Editors). Codon Publications, Brisbane, Australia. ISBN: 978-0-9944381-6-4; Doi: http://dx.doi. org/10.15586/codonpublications.parkinsonsdisease.2018 Copyright: The Authors. Licence: This open access article is licenced under Creative Commons Attribution 4.0 International (CC BY 4.0). https://creativecommons.org/licenses/by-nc/4.0/ 129 CP-005.indb 129 24/12/18 1:42 PM

------------------------------------------------------------------------------------------
HIT 2 | source=Red flags phenotyping.pdf | page=0 | distance=0.3304
knowledge and expertise to capture and interpret. Red flags involve (MSA), or at least hinted at an alternative diagnosis to Parkinson's different body parts and diverse aspects of the nervous system, often disease(PD) [1]. Sincethen, theter

In [28]:
test_questions = [
  "What are red flags that suggest atypical parkinsonism rather than idiopathic Parkinson's disease?",
  "How does essential tremor differ from Parkinson's disease tremor clinically?",
  "What clinical features distinguish progressive supranuclear palsy (PSP) from Parkinson's disease?",
  "What clinical features distinguish multiple system atrophy (MSA) from Parkinson's disease?",
  "What is drug-induced parkinsonism and how does it present compared to Parkinson's disease?",
  "What are common causes of secondary parkinsonism?",
  "What does bradykinesia mean clinically and how is it assessed?",
  "What does rigidity mean clinically and how is it assessed?",
  "What does the MDS-UPDRS Part III measure?",
  "What is Hoehn and Yahr staging used for?",
  "What non-motor symptoms are commonly associated with Parkinson's disease?",
  "When is dopamine transporter imaging (DaTscan) considered in evaluation?",
  "What are limitations of DaTscan/dopamine transporter imaging?",
  "What features suggest corticobasal syndrome rather than Parkinson's disease?",
  "What features suggest vascular parkinsonism rather than Parkinson's disease?",
  "What is the typical progression pattern of idiopathic Parkinson's disease?",
  "What gait abnormalities are described in Parkinson's disease?",
  "What does postural instability imply in parkinsonism evaluation?",
  "What early autonomic symptoms may suggest atypical parkinsonism?",
  "What does poor response to levodopa suggest about the diagnosis?"
]
print("Total questions:", len(test_questions))


Total questions: 20


In [29]:
for q in test_questions[:10]:
    print("\n" + "="*100)
    print("Q:", q)

    hits = retrieve_top_k(q, k=6)

    # show the top 3 retrievals (source/page + preview)
    for i, h in enumerate(hits[:3], 1):
        m = h["metadata"]
        print(f"\nTOP {i}: {m.get('source')} p{m.get('page')} | distance={h['distance']:.4f}")
        print(h["text"][:220].replace("\n", " "), "...")



Q: What are red flags that suggest atypical parkinsonism rather than idiopathic Parkinson's disease?

TOP 1: How to approach a patient with parkinsonism - red flags for atypical parkinsonism.pdf p28 | distance=0.3187
the accuracy of clinical diagnosis in Parkinson’s disease: A clinicopathologic study. Neurology,42(6),1142. Hughes,A.J.,Colosimo,C.,Kleedorfer,B.,Daniel,S.E.,&Lees,A.J.(1992).Thedopa- minergicresponseinmultiplesystematro ...

TOP 2: Recognizing Atypical Parkinsonisms.pdf p1 | distance=0.3212
reliable biomarkers have been established as diagnostic for any of the atypical parkinsonisms; however, ancillary brain imaging (as described in subsequent sections) is increasingly used as adjunctive diagnostic tools in ...

TOP 3: Parkinson’s Disease Pathogenesis and Clinical Aspects.pdf p142 | distance=0.3249
Aspects. Stoker TB, Greenland JC (Editors). Codon Publications, Brisbane, Australia. ISBN: 978-0-9944381-6-4; Doi: http://dx.doi. org/10.15586/codonpublications.parkinsonsdise

In [30]:
table_id = "table_red_flags_0001"

table_md = """
### Table: Red flags suggesting atypical parkinsonism (summary)
| Domain | Red flag (early or prominent) |
|---|---|
| Postural control | Recurrent early falls / early postural instability |
| Levodopa response | Poor or unsustained response to levodopa |
| Autonomic | Severe early orthostatic hypotension / urinary dysfunction |
| Eye movements | Supranuclear gaze palsy / abnormal vertical saccades |
"""

table_meta = {
    "source": "manual_table_red_flags",
    "page": 0,
    "modality": "table",
    "description": "Markdown table summarizing common red flags for atypical parkinsonism for retrieval testing.",
    "columns": "Domain, Red flag"
}

# Upsert behavior: delete if exists, then add
try:
    collection.delete(ids=[table_id])
except Exception:
    pass

collection.add(ids=[table_id], documents=[table_md.strip()], metadatas=[table_meta])

print("Added multimodal TABLE record. Collection count:", collection.count())


Added multimodal TABLE record. Collection count: 3055


In [35]:
hits = retrieve_top_k("table of red flags atypical parkinsonism", k=6)
show_hits(hits, max_chars=300)



------------------------------------------------------------------------------------------
HIT 1 | modality=table | source=manual_table_red_flags | page=0 | distance=0.2300
### Table: Red flags suggesting atypical parkinsonism (summary)
| Domain | Red flag (early or prominent) |
|---|---|
| Postural control | Recurrent early falls / early postural instability |
| Levodopa response | Poor or unsustained response to levodopa |
| Autonomic | Severe early orthostatic hypot

------------------------------------------------------------------------------------------
HIT 2 | modality=text | source=Recognizing Atypical Parkinsonisms.pdf | page=1 | distance=0.2895
reliable biomarkers have been established as diagnostic for any of the atypical parkinsonisms; however, ancillary brain imaging (as described in subsequent sections) is increasingly used as adjunctive diagnostic tools in select cases. While many features of the atypical parkinsonian syndromes overla

----------------------------------

In [36]:
%%writefile vector_db_utils.py
import chromadb
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction

DEFAULT_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

def get_client(persist_path: str):
    """Create/reuse a persistent Chroma client."""
    return chromadb.PersistentClient(path=persist_path)

def get_collection(client, name: str, model_name: str = DEFAULT_MODEL):
    """Get or create a collection with a fixed embedding function."""
    embed_fn = SentenceTransformerEmbeddingFunction(model_name=model_name)
    return client.get_or_create_collection(name=name, embedding_function=embed_fn)

def add_records(collection, ids, documents, metadatas=None):
    """Add new records."""
    if metadatas is None:
        metadatas = [{} for _ in ids]
    collection.add(ids=ids, documents=documents, metadatas=metadatas)

def query_records(collection, query_text: str, k: int = 5, where: dict | None = None):
    """Query top-k records. Optional metadata filter via where."""
    return collection.query(query_texts=[query_text], n_results=k, where=where)

def update_record(collection, record_id: str, new_document: str, new_metadata: dict | None = None):
    """Update an existing record by id."""
    # Chroma uses update() for existing IDs
    kwargs = {"ids": [record_id], "documents": [new_document]}
    if new_metadata is not None:
        kwargs["metadatas"] = [new_metadata]
    collection.update(**kwargs)

def upsert_record(collection, record_id: str, document: str, metadata: dict | None = None):
    """Upsert: add if new, update if exists."""
    kwargs = {"ids": [record_id], "documents": [document]}
    if metadata is not None:
        kwargs["metadatas"] = [metadata]
    collection.upsert(**kwargs)

def delete_records(collection, ids):
    """Delete records by ids."""
    collection.delete(ids=ids)


Writing vector_db_utils.py


In [37]:
%%writefile test_vector_ops.py
import os
import shutil
import pytest

from vector_db_utils import get_client, get_collection, add_records, query_records, update_record, upsert_record, delete_records

@pytest.fixture
def tmp_db_path(tmp_path):
    # each test run uses a fresh persistent directory
    return str(tmp_path / "chroma_test_db")

@pytest.fixture
def collection(tmp_db_path):
    client = get_client(tmp_db_path)
    col = get_collection(client, name="test_collection")
    return col

def test_1_collection_starts_empty(collection):
    assert collection.count() == 0

def test_2_add_records_increases_count(collection):
    add_records(collection,
                ids=["a1","a2"],
                documents=["parkinson red flags early falls", "essential tremor vs parkinson tremor"],
                metadatas=[{"modality":"text"},{"modality":"text"}])
    assert collection.count() == 2

def test_3_query_returns_results(collection):
    add_records(collection,
                ids=["b1","b2"],
                documents=["poor levodopa response suggests atypical", "UPDRS part III motor exam"],
                metadatas=[{"topic":"atypical"},{"topic":"scale"}])
    res = query_records(collection, "levodopa response", k=2)
    assert len(res["documents"][0]) >= 1
    assert isinstance(res["documents"][0][0], str)

def test_4_query_returns_metadata(collection):
    add_records(collection,
                ids=["c1"],
                documents=["Hoehn and Yahr staging describes disease severity"],
                metadatas=[{"topic":"staging"}])
    res = query_records(collection, "Hoehn and Yahr", k=1)
    assert "metadatas" in res
    assert res["metadatas"][0][0]["topic"] == "staging"

def test_5_where_filter_works(collection):
    add_records(collection,
                ids=["d1","d2"],
                documents=["table red flags atypical parkinsonism", "random unrelated text"],
                metadatas=[{"modality":"table"},{"modality":"text"}])
    res = query_records(collection, "red flags", k=5, where={"modality":"table"})
    assert len(res["documents"][0]) >= 1
    assert res["metadatas"][0][0]["modality"] == "table"

def test_6_update_record_changes_text(collection):
    add_records(collection, ids=["e1"], documents=["old text"], metadatas=[{"v":1}])
    update_record(collection, "e1", new_document="new updated text", new_metadata={"v":2})
    res = query_records(collection, "updated", k=1)
    assert "updated" in res["documents"][0][0]
    assert res["metadatas"][0][0]["v"] == 2

def test_7_upsert_adds_when_missing(collection):
    upsert_record(collection, "f1", "fresh upsert text", {"v":1})
    assert collection.count() == 1

def test_8_upsert_updates_when_exists(collection):
    upsert_record(collection, "g1", "version one", {"v":1})
    upsert_record(collection, "g1", "version two", {"v":2})
    res = query_records(collection, "version two", k=1)
    assert "version two" in res["documents"][0][0]
    assert res["metadatas"][0][0]["v"] == 2

def test_9_delete_removes_records(collection):
    add_records(collection, ids=["h1","h2"], documents=["x","y"], metadatas=[{},{}])
    delete_records(collection, ["h1"])
    assert collection.count() == 1


Writing test_vector_ops.py


In [38]:
!pip -q install pytest
!pytest -q test_vector_ops.py


........F                                                                [100%]
=================================== FAILURES ===================================
________________________ test_9_delete_removes_records _________________________

collection = Collection(name=test_collection)

    def test_9_delete_removes_records(collection):
>       add_records(collection, ids=["h1","h2"], documents=["x","y"], metadatas=[{},{}])

test_vector_ops.py:74: 
_ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ 
vector_db_utils.py:19: in add_records
    collection.add(ids=ids, documents=documents, metadatas=metadatas)
/usr/local/lib/python3.12/dist-packages/chromadb/api/models/Collection.py:97: in add
    add_request = self._validate_and_prepare_add_request(
/usr/local/lib/python3.12/dist-packages/chromadb/api/models/CollectionCommon.py:103: in wrapper
    return func(self, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^
/usr/local/lib/python3.12/dist-packages

In [39]:
%%writefile test_rag_retrieval.py
import os
import pytest

from vector_db_utils import get_client, get_collection, query_records

# CHANGE THIS to match your persistent DB path on Drive if running locally.
# In Colab, you can point it to your Drive path.
PERSIST_PATH = "/content/drive/MyDrive/RAG/chroma_db"
COLLECTION_NAME = "pd_rag_chunks"

QUESTIONS = [
    "What are red flags for atypical parkinsonism?",
    "How does essential tremor differ from Parkinson's disease tremor clinically?",
    "What features suggest progressive supranuclear palsy (PSP)?",
    "What features suggest multiple system atrophy (MSA)?",
    "What is drug-induced parkinsonism?",
    "What does MDS-UPDRS Part III measure?",
    "What is Hoehn and Yahr staging used for?",
    "When is DaTscan considered in evaluation?",
    "What are limitations of DaTscan?",
    "What does poor response to levodopa suggest about the diagnosis?",
    "What gait abnormalities are described in Parkinson's disease?",
    "What early autonomic symptoms may suggest atypical parkinsonism?"
]

def test_retrieval_returns_k_hits():
    client = get_client(PERSIST_PATH)
    collection = get_collection(client, COLLECTION_NAME)

    # Require that collection is populated
    assert collection.count() > 0, "Collection is empty. Did you run chunk insertion first?"

    for q in QUESTIONS[:12]:  # 10–15 questions; using 12 here
        res = query_records(collection, q, k=6)
        docs = res["documents"][0]
        metas = res["metadatas"][0]

        assert len(docs) >= 5, f"Too few hits for question: {q}"
        assert len(docs[0].strip()) > 50, f"Top hit too short/empty for question: {q}"

        # metadata sanity checks
        assert "source" in metas[0], f"Missing source metadata for: {q}"
        assert "page" in metas[0], f"Missing page metadata for: {q}"
        assert "modality" in metas[0], f"Missing modality metadata for: {q}"


Writing test_rag_retrieval.py


In [40]:
!pytest -q test_rag_retrieval.py


.                                                                        [100%]
1 passed in 16.63s
